In [2]:


import os,cv2
import numpy as np
from matplotlib import pyplot as plt


In [3]:
videoPath = "Robots.mp4"
print("File exists:", os.path.exists(videoPath))  

# With OpenCV we can load a video using the cv2.VideoCapture function:
cap = cv2.VideoCapture(videoPath)

if not cap.isOpened():
    print("Error: Could not open video.")
    exit()
# When we need a single frame of the video, we can then use the read function of the created cv2.VideoCapture object:

while True:

    # The variable ret will be either True or False indicating if the frame was read correctly. 
    # The variable frame will then be the first frame of the video. If we call the same function again, it will return the second frame of the video, and so on.
    ret, frame = cap.read()

    if not ret:
        break
    

        

    cv2.imshow("IMAGE", frame)
    
    if cv2.waitKey(20) & 0xFF ==ord ('q'):
        break
    

# cv2.imshow("IMAGE", frame)
cap.release()
cv2.destroyAllWindows()

File exists: True


In [7]:
import cv2
import numpy as np
import os

# --- Check if video file exists ---
videoPath = "Robots.mp4"
print("File Exists:", os.path.exists(videoPath))

# --- Open video ---
cap = cv2.VideoCapture(videoPath)
if not cap.isOpened():
    print("Error: Could Not Open File")
    exit()

# --- Read the first frame ---
ret, old_frame = cap.read()
if not ret:
    print("Error: Frame is not read")
    exit()

# Convert to grayscale
gray1 = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

# Detect initial good features
feat1 = cv2.goodFeaturesToTrack(gray1, maxCorners=200, qualityLevel=0.3, minDistance=5)

# Green color for tracking box (BGR)
color = (0, 255, 0)

# Minimum motion threshold in pixels
motion_threshold = 8

while True:
    ret, new_frame = cap.read()
    if not ret:
        print("End of Video or Frame Not Read")
        break

    # Convert to grayscale
    gray2 = cv2.cvtColor(new_frame, cv2.COLOR_BGR2GRAY)

    # Calculate optical flow
    feat2, status, error = cv2.calcOpticalFlowPyrLK(gray1, gray2, feat1, None)

    if feat2 is not None and status is not None:
        # Select good points
        good_new = feat2[status == 1]
        good_old = feat1[status == 1]

        moving_points_new = []
        x_coords, y_coords = [], []

        for new, old in zip(good_new, good_old):
            x_new, y_new = new.ravel()
            x_old, y_old = old.ravel()

            # Compute motion distance
            dist = np.sqrt((x_new - x_old) ** 2 + (y_new - y_old) ** 2)

            if dist > motion_threshold:
                # Collect coordinates of moving points
                x_coords.append(int(x_new))
                y_coords.append(int(y_new))
                moving_points_new.append([[x_new, y_new]])

        # If we have moving points, draw one big bounding box
        if x_coords and y_coords:
            x_min, x_max = min(x_coords), max(x_coords)
            y_min, y_max = min(y_coords), max(y_coords)
            cv2.rectangle(new_frame, (x_min, y_min), (x_max, y_max), color, 2)

        # Update features
        if moving_points_new:
            feat1 = np.array(moving_points_new, dtype=np.float32)
        else:
            feat1 = cv2.goodFeaturesToTrack(gray2, maxCorners=200, qualityLevel=0.3, minDistance=5)

    # Update the previous frame
    gray1 = gray2.copy()

    # Show frame
    cv2.imshow("Moving Robot Tracking", new_frame)

    # Press 'q' to quit
    if cv2.waitKey(20) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()


File Exists: True


In [6]:
import cv2
import numpy as np
import os

# --- Check if video file exists ---
videoPath = "Robots.mp4"
print("File Exists:", os.path.exists(videoPath))

# --- Open video ---
cap = cv2.VideoCapture(videoPath)
if not cap.isOpened():
    print("Error: Could Not Open File")
    exit()

# --- Read the first frame ---
ret, old_frame = cap.read()
if not ret:
    print("Error: Frame is not read")
    exit()

# Convert to grayscale
gray1 = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

# Detect initial good features
feat1 = cv2.goodFeaturesToTrack(gray1, maxCorners=500, qualityLevel=0.3, minDistance=5)

# Green color for tracking points (BGR)
color = (0, 255, 0)

# Minimum motion threshold in pixels
motion_threshold = 2

# Frame counter for periodic refresh
frame_count = 0

while True:
    ret, new_frame = cap.read()
    if not ret:
        print("End of Video or Frame Not Read")
        break

    # Convert to grayscale
    gray2 = cv2.cvtColor(new_frame, cv2.COLOR_BGR2GRAY)

    # Calculate optical flow
    feat2, status, error = cv2.calcOpticalFlowPyrLK(gray1, gray2, feat1, None)

    # Select good points
    good_new = feat2[status == 1]
    good_old = feat1[status == 1]

    moving_points_new = []
    for new, old in zip(good_new, good_old):
        x_new, y_new = new.ravel()
        x_old, y_old = old.ravel()

        # Compute motion distance
        dist = np.sqrt((x_new - x_old)**2 + (y_new - y_old)**2)

        # Keep only moving points
        if dist > motion_threshold:
            cv2.circle(new_frame, (int(x_new), int(y_new)), 4, color, -1)
            cv2.line(new_frame, (int(x_old), int(y_old)), (int(x_new), int(y_new)), color, 2)
            moving_points_new.append([[x_new, y_new]])

    # Update features
    if moving_points_new:
        feat1 = np.array(moving_points_new, dtype=np.float32)
    else:
        # Re-detect features if none remain
        feat1 = cv2.goodFeaturesToTrack(gray2, maxCorners=500, qualityLevel=0.3, minDistance=5)

    # Refresh features every 20 frames
    frame_count += 1
    if frame_count % 20 == 0:
        feat1 = cv2.goodFeaturesToTrack(gray2, maxCorners=500, qualityLevel=0.3, minDistance=5)

    # Update previous frame
    gray1 = gray2.copy()

    # Show frame
    cv2.imshow("Moving Robot Tracking", new_frame)

    # Press 'q' to quit
    if cv2.waitKey(20) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()


File Exists: True
End of Video or Frame Not Read
